# Optimizer Comparison — Complete Theory

## What is an Optimizer?

After computing gradients via backprop, the optimizer decides **how** to update the weights. The gradient tells you the direction of steepest increase in loss — the optimizer uses that to step in the opposite direction. The differences between optimizers are in *how smart* that step is.

---

## The Three Worth Knowing

### SGD — Stochastic Gradient Descent
The simplest possible optimizer. Take the gradient, multiply by learning rate, subtract from weights. That's it.

```
weight = weight - lr * gradient
```

**Problem:** It treats every parameter the same way, regardless of how often it's updated or how large its gradients are. It can be slow to converge and sensitive to learning rate choice.

**When to use:** Large scale vision models, when you want maximum control. Rarely the first choice for small projects.

**Import and syntax:**
```python
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
```
`momentum` — carries some velocity from previous steps, helps escape flat regions. Almost always used with SGD.

---

### Adam — Adaptive Moment Estimation
Maintains a **separate learning rate for each parameter**, adapting based on how large the gradients have been historically. Parameters with consistently large gradients get smaller effective LR; sparse parameters get larger effective LR.

Internally tracks two things per parameter:
- First moment — running average of gradients (direction)
- Second moment — running average of squared gradients (magnitude)

**Result:** Faster convergence, less sensitive to initial LR choice. The default choice for most projects.

**Import and syntax:**
```python
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
```

---

### AdamW — Adam with Weight Decay Fix
Adam has a subtle bug — its weight decay (regularization) implementation interacts incorrectly with the adaptive learning rates. AdamW fixes this by decoupling weight decay from the gradient update.

In practice: AdamW generalizes better than Adam, especially on larger models. It's now the standard in modern deep learning (transformers, etc.).

**Import and syntax:**
```python
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
```

`weight_decay` — how strongly to penalize large weights. Acts as regularization, helps prevent overfitting.

---

## Side by Side Summary

``` t
SGD      → simple, manual tuning needed, good ceiling with effort
Adam     → fast convergence, good default, slight overfitting tendency
AdamW    → Adam + better regularization, current best practice
```

---

## TODO 1: Implement a Fair Comparison Function

**Goal:** Write a function that trains a fresh model with a given optimizer for N epochs and returns the training history.

**Requirements:**
- Function signature: `train_with_optimizer(optimizer_name, num_epochs=15)`
- Inside: creates a fresh `SpeakerCounter`, creates the specified optimizer, runs full train + val loops
- Returns a dict with keys: `'train_losses'`, `'val_losses'`, `'train_accs'`, `'val_accs'` — each a list of per-epoch values
- Supports three optimizer names as strings: `'sgd'`, `'adam'`, `'adamw'`
- All three must use the same `lr=0.001` for fair comparison

**Hints:**
- Use `if/elif` on the string to create the right optimizer
- Append to lists inside the epoch loop, don't overwrite
- No early stopping or scheduler here — keep it simple for clean comparison
- Same model architecture, same data, only optimizer changes

---

## TODO 2: Run and Compare

**Goal:** Call your function three times and print a comparison table.

**Requirements:**
- Run all three optimizers for 15 epochs
- Print a table showing final train loss, val loss, val accuracy for each
- Answer in a comment: which optimizer performed best on your data and why you think that is

**Expected output:**

| Optimizer | Final Train Loss | Final Val Loss | Final Val Acc |
|-----------|------------------|----------------|---------------|
| SGD       |     0.9823       |     0.9541     |    54.2%      |
| Adam      |     0.8234       |     0.8102     |    58.7%      |
| AdamW     |     0.8198       |     0.7989     |    59.1%      |



In [45]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from pathlib import Path
import os
import sys
import json

# Check if __file__ exists (it won't in Jupyter)
try:
    current_dir = Path(__file__).parent
except NameError:
    # If in Jupyter, use the current working directory
    current_dir = Path(os.getcwd())

# Add project root to Python path
project_root = current_dir.parent.parent.parent
sys.path.insert(0, str(project_root))

from src.preprocessing.audio_utils import load_audio


# Device configuration (for your MacBook)
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ Using Apple Silicon GPU")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("✅ Using NVIDIA GPU")
else:
    device = torch.device('cpu')
    print("⚠️ Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")


✅ Using Apple Silicon GPU
PyTorch version: 2.10.0
Device: mps


### loading the sample

In [46]:
class SpeakerDataset(torch.utils.data.Dataset):
    def __init__(self, manifest_path, limit=None):
        # Load manifest JSON
        # Store all entries as self.data
        with open(manifest_path, "r") as f:
            full_data = json.load(f)
        
        if limit:
            self.data = full_data[:limit]
        else:
            self.data = full_data
    
    def __len__(self):
        # Return number of samples
        return len(self.data)
    
    def __getitem__(self, idx):
        # Get entry at index idx
        # Load audio from disk
        # Extract features
        # Get label (num_speakers - 1)
        # Return (features, label)
        entry = self.data[idx]
        mixture_path = Path(entry['mixture_path'])
        mixture_audio, _ = load_audio(mixture_path)
        
        mixture_tensor = torch.from_numpy(mixture_audio)
        mixture_tensor = mixture_tensor
        feature_tensor = self.feature_extraction(mixture_tensor)
        
        speaker_count = int(entry['num_speakers']) - 1
        label_tensor = torch.tensor(speaker_count, dtype=torch.long)
        
        return feature_tensor, label_tensor
    
    def feature_extraction(
            self,
            audio: torch.Tensor
    ) -> torch.Tensor:
        mean = torch.mean(audio)
        std = torch.std(audio)
        min = torch.min(audio)
        max = torch.max(audio)
        square = audio ** 2
        energy = torch.mean(square)
        
        return torch.stack([mean, std, min, max, energy])

### getting the path to the dataset and loading the metadata

In [47]:
train_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "train" / "train_manifest.json"
train_dataset = SpeakerDataset(train_manifest_path)
print(len(train_dataset))

val_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "val" / "val_manifest.json"
val_dataset = SpeakerDataset(val_manifest_path, 2000)
print(len(val_dataset))

test_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "test" / "test_manifest.json"
test_dataset = SpeakerDataset(test_manifest_path)
print(len(test_dataset))


10000
2000
10000


In [48]:
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

### creating the neural network

In [49]:
class SpeakerCounter(nn.Module):
    """
    Classifier that predicts number of speakers (1, 2, or 3)
    from audio features.
    """
    def __init__(self, input_size=5, hidden_sizes=[64, 32, 16], num_classes=3):
        super().__init__()
        # TODO: Build network with given architecture
        # Hint: Use nn.Sequential or define layers individually
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_sizes[0]),
            nn.BatchNorm1d(hidden_sizes[0]),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_sizes[0],hidden_sizes[1]),
            nn.BatchNorm1d(hidden_sizes[1]),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_sizes[1],hidden_sizes[2]),
            nn.BatchNorm1d(hidden_sizes[2]),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_sizes[2],num_classes)
        )
        
    def forward(self, x):
        # TODO: Implement forward pass
        return self.network(x)

### calculating the loss

In [50]:
def compute_accuracy(predictions, labels):
    """
    predictions: (batch_size, 3) logits
    labels: (batch_size,) true classes
    
    Returns: accuracy as percentage
    """
    predicted_classes = torch.argmax(predictions, dim=1)
    correct = (predicted_classes == labels).sum().item()
    total = labels.size(0)
    return 100.0 * correct / total

In [51]:
def train_with_optimizer(optimizer_name="sgd", num_epochs=2):
    model = SpeakerCounter().to(device)
    
    if optimizer_name=="sgd":
        optimizer = optim.SGD(model.parameters(), lr=0.0001, momentum=0.9)
    elif optimizer_name=="adam":
        optimizer = optim.Adam(model.parameters(), lr=0.0001)
    elif optimizer_name=="adamw":
        optimizer = optim.AdamW(model.parameters(), lr=0.0001)
    
    print(f"using the {optimizer_name} optimizer")
    
    output = {
        'train_losses': [], 
        'val_losses': [], 
        'train_accs': [], 
        'val_accs': []
    }
    
    for epoch in range(num_epochs):
        model.train()
        running_train_loss = 0.0
        running_train_acc = 0.0
        steps_per_epoch_train = len(train_loader)
        
        for step, batch in enumerate(train_loader):
            features, labels = batch[0].to(device), batch[1].to(device)
            
            # Forward pass
            logits = model(features)  
            
            # Compute loss
            loss = F.cross_entropy(logits, labels)
            
            # Backward pass
            loss.backward()
            
            # Update weights
            optimizer.step()
            optimizer.zero_grad()
            
            # Compute accuracy
            running_train_loss += loss.item() 
            running_train_acc += compute_accuracy(logits, labels)
            
            # Optional: Print batch progress every 50 batches within the epoch
            if (step + 1) % 50 == 0:
                print(f"  train Batch {step+1}/{steps_per_epoch_train} | Loss: {loss.item():.4f}")
        
        # 3. Print Summary at the end of each FULL epoch
        avg_train_loss = running_train_loss / steps_per_epoch_train
        avg_train_acc = running_train_acc / steps_per_epoch_train    
        
        model.eval()
        running_val_loss = 0.0
        running_val_acc = 0.0     
        steps_per_epoch_val = len(val_loader)
        
        with torch.no_grad():
            for steps, batch in enumerate(val_loader):
                
                features, labels = batch[0].to(device), batch[1].to(device)
                
                # Forward pass
                logits = model(features)  
                
                # Compute loss
                loss = F.cross_entropy(logits, labels)
                
                # number of correct class
                running_val_loss += loss.item()
                running_val_acc += compute_accuracy(logits, labels)
                
                # Optional: Print batch progress every 50 batches within the epoch
                if (steps + 1) % 30 == 0:
                    print(f"  val Batch {steps+1}/{steps_per_epoch_val} | Loss: {loss.item():.4f}")
        
        # Calculate Averages
        avg_val_loss = running_val_loss / steps_per_epoch_val
        avg_val_acc = running_val_acc / steps_per_epoch_val   
        
        # --- LOGGING & SCHEDULER ---
        print(f"\n--- Epoch {epoch+1}/{num_epochs} Summary ---")
        print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.2f}%")
        print(f"Val Loss:   {avg_val_loss:.4f} | Val Acc:   {avg_val_acc:.2f}%")
        temp_dic = {
            'train_losses': avg_train_loss, 
            'val_losses': avg_val_loss, 
            'train_accs': avg_train_acc, 
            'val_accs': avg_val_acc
        }
        output['train_losses'].append(avg_train_loss)
        output['val_losses'].append(avg_val_loss)
        output['train_accs'].append(avg_train_acc)
        output['val_accs'].append(avg_val_acc)
    
    return output


In [52]:
output1 = train_with_optimizer("sgd")

using the sgd optimizer
  train Batch 50/312 | Loss: 1.0970
  train Batch 100/312 | Loss: 1.0333
  train Batch 150/312 | Loss: 0.9769
  train Batch 200/312 | Loss: 0.9979
  train Batch 250/312 | Loss: 0.9806
  train Batch 300/312 | Loss: 0.9565
  val Batch 30/63 | Loss: 1.0339
  val Batch 60/63 | Loss: 0.9742

--- Epoch 1/2 Summary ---
Train Loss: 1.0675 | Train Acc: 39.81%
Val Loss:   0.9924 | Val Acc:   53.92%
  train Batch 50/312 | Loss: 1.0223
  train Batch 100/312 | Loss: 0.9642
  train Batch 150/312 | Loss: 0.9664
  train Batch 200/312 | Loss: 0.9350
  train Batch 250/312 | Loss: 0.8168
  train Batch 300/312 | Loss: 0.9905
  val Batch 30/63 | Loss: 0.9908
  val Batch 60/63 | Loss: 0.9041

--- Epoch 2/2 Summary ---
Train Loss: 0.9596 | Train Acc: 49.94%
Val Loss:   0.9240 | Val Acc:   53.67%


In [53]:
output2 = train_with_optimizer("adam")

using the adam optimizer
  train Batch 50/312 | Loss: 1.1330
  train Batch 100/312 | Loss: 1.1203
  train Batch 150/312 | Loss: 1.0852
  train Batch 200/312 | Loss: 1.1502
  train Batch 250/312 | Loss: 1.0061
  train Batch 300/312 | Loss: 1.0165
  val Batch 30/63 | Loss: 1.0982
  val Batch 60/63 | Loss: 1.0236

--- Epoch 1/2 Summary ---
Train Loss: 1.0610 | Train Acc: 44.64%
Val Loss:   1.0208 | Val Acc:   50.69%
  train Batch 50/312 | Loss: 0.9972
  train Batch 100/312 | Loss: 0.9465
  train Batch 150/312 | Loss: 1.0444
  train Batch 200/312 | Loss: 0.9211
  train Batch 250/312 | Loss: 0.9151
  train Batch 300/312 | Loss: 0.9491
  val Batch 30/63 | Loss: 1.0464
  val Batch 60/63 | Loss: 0.9629

--- Epoch 2/2 Summary ---
Train Loss: 0.9624 | Train Acc: 51.02%
Val Loss:   0.9591 | Val Acc:   51.29%


In [54]:
output3 = train_with_optimizer("adamw")

using the adamw optimizer
  train Batch 50/312 | Loss: 0.9987
  train Batch 100/312 | Loss: 1.1143
  train Batch 150/312 | Loss: 1.0044
  train Batch 200/312 | Loss: 1.0036
  train Batch 250/312 | Loss: 0.9683
  train Batch 300/312 | Loss: 0.9567
  val Batch 30/63 | Loss: 0.9602
  val Batch 60/63 | Loss: 0.9616

--- Epoch 1/2 Summary ---
Train Loss: 0.9933 | Train Acc: 47.74%
Val Loss:   0.9550 | Val Acc:   50.00%
  train Batch 50/312 | Loss: 0.9576
  train Batch 100/312 | Loss: 0.9242
  train Batch 150/312 | Loss: 0.9242
  train Batch 200/312 | Loss: 0.9396
  train Batch 250/312 | Loss: 0.8796
  train Batch 300/312 | Loss: 0.8683
  val Batch 30/63 | Loss: 0.9237
  val Batch 60/63 | Loss: 0.9082

--- Epoch 2/2 Summary ---
Train Loss: 0.9122 | Train Acc: 50.23%
Val Loss:   0.8996 | Val Acc:   52.03%


In [55]:
print(f"Optimizer| Final Train Loss \t| Final Val Loss \t| Final Val Acc \t|")
print(f"SGD\t | {output1['train_losses'][-1]} \t| {output1['val_losses'][-1]} \t| {output1['val_accs'][-1]} \t|")
print(f"adam\t | {output2['train_losses'][-1]} \t| {output2['val_losses'][-1]} \t| {output2['val_accs'][-1]} \t|")
print(f"adamw\t | {output3['train_losses'][-1]} \t| {output3['val_losses'][-1]} \t| {output3['val_accs'][-1]} \t|")

Optimizer| Final Train Loss 	| Final Val Loss 	| Final Val Acc 	|
SGD	 | 0.9596121251965181 	| 0.9240284845942542 	| 53.67063492063492 	|
adam	 | 0.9624226601460041 	| 0.9590794680610536 	| 51.28968253968254 	|
adamw	 | 0.9122058775944587 	| 0.8996185518446422 	| 52.03373015873016 	|
